# NB2 — Core-7 Drop, Clean Positives & Item Metadata

Notebook này thực hiện **category drop trước negative sampling** và giữ lại taxonomy cần thiết cho bước sau:

1. dùng official split `train/valid/test` của Polyvore1000;
2. map `master_category` thành `TOP/BOTTOM/DRESS/OUTERWEAR/SHOES/BAG/HAT/DROP`;
3. loại item được map thành `DROP`;
4. đếm lại số item của outfit;
5. giữ outfit còn ít nhất 3 item;
6. xuất `category_clean_{split}.jsonl`;
7. xuất **versioned Core-7 item metadata** chứa `master_category` + `coarse_category` cho đúng các item đã serialize.

> Đây vẫn là output trung gian, chưa phải dataset `READY_TO_TRAIN`. Sau category drop còn phải kiểm tra ảnh/embedding, drop item lỗi và đếm lại độ dài outfit. Notebook **không tạo negative**; negative cũ không được dùng lại sau khi composition của outfit thay đổi.


## Luồng xử lý

```text
Polyvore1000 positive outfits
        ↓
master_category → Core-7 hoặc DROP
        ↓
loại item DROP
        ↓
recompute outfit length
        ↓
giữ outfit length >= 3
        ↓
        ├── category_clean_{split}.jsonl
        └── core7_item_metadata_v1_{split}.jsonl
                item_id
                source_kit_id
                slot_index
                master_category
                coarse_category
                split
                item_metadata_version
                category_mapping_version
        ↓
image / embedding validation
        ↓
final clean positives + metadata đã lọc tương ứng
        ↓
negative sampler mới
```

`category_clean_*.jsonl` chỉ giữ `item_id` trong sample để scorer-ready schema gọn. Taxonomy không bị mất nữa vì được serialize riêng trong item metadata artifact, lookup bằng `item_id`.


## 1. Cài thư viện

Chỉ cần `datasets` để đọc Polyvore1000 từ Hugging Face. Không chạy FashionCLIP trong notebook này.


In [ ]:
%pip install -q "datasets>=3,<5"

## 2. Tìm repository và cấu hình runtime portable

- Local/VS Code: mở notebook từ repository đã clone; mặc định artifact nằm trong `./data`.
- Colab: clone repo hoặc bật `AUTO_CLONE_REPO=True`, mount Drive nếu muốn, rồi set `FASHION_ARTIFACT_ROOT`.
- Core code không import `google.colab` và không hard-code `/content/drive`.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/ThinhTran2208/opisoverated.git"
AUTO_CLONE_REPO = False  # Chỉ bật khi runtime chưa có repository, ví dụ Colab mới.


def find_repo_root(start: Path = Path.cwd()):
    explicit = os.environ.get("FASHION_PROJECT_ROOT")
    if explicit:
        candidate = Path(explicit).expanduser().resolve()
        if (candidate / "src/data/prepare_core7_dataset.py").exists():
            return candidate
        raise FileNotFoundError(f"FASHION_PROJECT_ROOT không hợp lệ: {candidate}")

    current = start.expanduser().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src/data/prepare_core7_dataset.py").exists():
            return candidate
    return None


REPO_ROOT = find_repo_root()
if REPO_ROOT is None and AUTO_CLONE_REPO:
    REPO_ROOT = (Path.cwd() / "opisoverated").resolve()
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)

if REPO_ROOT is None:
    raise RuntimeError(
        "Không tìm thấy repository. Hãy mở notebook từ repo đã clone, "
        "set FASHION_PROJECT_ROOT, hoặc bật AUTO_CLONE_REPO=True."
    )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data.runtime_paths import load_runtime_paths

RUNTIME_PATHS = load_runtime_paths(repo_root=REPO_ROOT)
print("Repo root      :", REPO_ROOT)
print("Path config    :", RUNTIME_PATHS.config_path)
print("Artifact root  :", RUNTIME_PATHS.artifact_root)

from src.data.prepare_core7_dataset import (
    CORE_CATEGORIES,
    CORE7_ITEM_METADATA_VERSION,
    load_category_mapping,
    prepare_clean_positive_split,
)

print("Core categories      :", CORE_CATEGORIES)
print("Item metadata version:", CORE7_ITEM_METADATA_VERSION)

## 3. Cấu hình output

`CORE7_OUTPUT_DIR` được resolve theo thứ tự:

1. environment variable;
2. `configs/data_paths.local.json`;
3. `configs/data_paths.example.json`.

Không cần Drive khi chạy local. Debug vẫn dùng 50 outfit đầu của train.

In [ ]:
CORE7_OUTPUT_DIR = RUNTIME_PATHS.core7_dir
CORE7_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAPPING_PATH = REPO_ROOT / "configs/category_mapping_core7_v1.json"
MIN_ITEMS = 3

DEBUG_SPLIT = "train"
DEBUG_LIMIT = 50
DEBUG_POSITIVE_OUTPUT = CORE7_OUTPUT_DIR / "debug_category_clean_train.jsonl"
DEBUG_METADATA_OUTPUT = CORE7_OUTPUT_DIR / "debug_core7_item_metadata_v1_train.jsonl"

print("Mapping        :", MAPPING_PATH)
print("Core-7 output  :", CORE7_OUTPUT_DIR)
print("Positive output:", DEBUG_POSITIVE_OUTPUT)
print("Metadata output:", DEBUG_METADATA_OUTPUT)

## 4. Kiểm tra mapping Core-7/DROP

Mapping hiện có version riêng. `master_category` là taxonomy nguồn; `coarse_category` là taxonomy Core-7 của project. Hai field này phải cùng tồn tại trong metadata artifact, không overwrite nhau.


In [ ]:
mapping_metadata, category_mapping = load_category_mapping(MAPPING_PATH)

decision_counts = {}
for decision in category_mapping.values():
    decision_counts[decision] = decision_counts.get(decision, 0) + 1

print("Mapping version:", mapping_metadata["mapping_version"])
print("Mapping status :", mapping_metadata["status"])
print("Master categories:", len(category_mapping))
print("Decision counts:")
for decision, count in sorted(decision_counts.items()):
    print(f"  {decision:10s}: {count}")

print("\nVí dụ category được giữ:")
shown = 0
for master, coarse in category_mapping.items():
    if coarse != "DROP":
        print(f"  {master:35s} -> {coarse}")
        shown += 1
        if shown == 20:
            break


## 5. Chạy thử trên 50 outfit train

Core code sẽ:

- load item/kit metadata của official train split;
- kiểm tra mapping cover toàn bộ `master_category`;
- drop item theo mapping;
- tạo positive sample canonical;
- drop outfit còn dưới 3 item;
- build metadata cho **đúng item xuất hiện trong positive output**;
- validate exact coverage giữa sample item IDs và metadata item IDs;
- ghi cả hai JSONL vào Drive.


In [ ]:
debug_report = prepare_clean_positive_split(
    split=DEBUG_SPLIT,
    output_path=DEBUG_POSITIVE_OUTPUT,
    item_metadata_output_path=DEBUG_METADATA_OUTPUT,
    mapping_path=MAPPING_PATH,
    min_items=MIN_ITEMS,
    debug_limit=DEBUG_LIMIT,
)

debug_report


## 6. Xem positive sample sau khi drop

Positive output vẫn giữ canonical scorer schema:

- `sample_id = <kit_id>_pos`;
- `source_kit_id`;
- `paired_positive_sample_id = null`;
- `items` chỉ là list `item_id`;
- `label = 1`;
- `negative_metadata = null`.

Category không nhét lại vào sample; category lookup nằm ở artifact riêng.


In [ ]:
def read_jsonl_head(path: Path, n: int = 5):
    rows = []
    with path.open("r", encoding="utf-8") as stream:
        for index, line in enumerate(stream):
            if index >= n:
                break
            rows.append(json.loads(line))
    return rows


for sample in read_jsonl_head(DEBUG_POSITIVE_OUTPUT, n=5):
    print(json.dumps(sample, ensure_ascii=False, indent=2))
    print("-" * 80)


## 7. Xem Core-7 item metadata artifact

Mỗi row là một lookup record theo `item_id`:

```json
{
  "item_metadata_version": "core7-item-metadata-v1",
  "category_mapping_version": "core7-v1",
  "split": "train",
  "item_id": "...",
  "source_kit_id": "...",
  "slot_index": 1,
  "master_category": "...",
  "coarse_category": "TOP"
}
```

`category_mapping_version` ghi exact mapping đã dùng để tạo `coarse_category`. Khi mapping đổi version, metadata artifact mới phải được regenerate thay vì overwrite ý nghĩa artifact cũ.


In [ ]:
metadata_head = read_jsonl_head(DEBUG_METADATA_OUTPUT, n=10)
for row in metadata_head:
    print(json.dumps(row, ensure_ascii=False, indent=2))
    print("-" * 80)


## 8. Sanity check lookup positive → metadata

Cell này minh họa đúng interface mà image-validation và negative sampler mới sẽ cần: `item_id → metadata`.


In [ ]:
metadata_by_item = {}
with DEBUG_METADATA_OUTPUT.open("r", encoding="utf-8") as stream:
    for line in stream:
        row = json.loads(line)
        metadata_by_item[row["item_id"]] = row

sample = read_jsonl_head(DEBUG_POSITIVE_OUTPUT, n=1)[0]
print("sample_id:", sample["sample_id"])
for item_id in sample["items"]:
    meta = metadata_by_item[item_id]
    print(
        item_id,
        "| master=", meta["master_category"],
        "| coarse=", meta["coarse_category"],
    )


## 9. Đọc nhanh report

Hai validation phải pass:

- `validation.pass`: positive sample contract;
- `item_metadata_validation.pass`: metadata schema + version + exact item coverage.

Debug chỉ lấy 50 outfit cho sample/metadata output. Mapping coverage và category-drop item statistics vẫn được tính trên metadata của cả split.


In [ ]:
summary = {
    "items_before_drop": debug_report["filter"]["raw_item_count"],
    "items_after_drop": debug_report["filter"]["kept_item_count"],
    "items_dropped": debug_report["filter"]["dropped_item_count"],
    "debug_outfits_processed": debug_report["outfits"]["kits_processed"],
    "debug_outfits_kept": debug_report["outfits"]["outfits_kept"],
    "debug_outfits_dropped": debug_report["outfits"]["outfits_dropped_below_min_items"],
    "positive_validation_pass": debug_report["validation"]["pass"],
    "metadata_item_count": debug_report["item_metadata_validation"]["item_count"],
    "metadata_validation_pass": debug_report["item_metadata_validation"]["pass"],
    "item_metadata_version": debug_report["item_metadata_version"],
    "category_mapping_version": debug_report["mapping_version"],
}

print(json.dumps(summary, ensure_ascii=False, indent=2))


## 10. Full run — chỉ bật sau khi nhóm review mapping

Mặc định `RUN_FULL=False` để tránh vô tình regenerate full dataset.

Sau khi nhóm review `configs/category_mapping_core7_v1.json`:

1. đổi mapping `status` từ `draft` thành `frozen`;
2. đổi `RUN_FULL=True`;
3. chạy cell này.

Mỗi split sẽ có 3 file: positive JSONL, versioned item metadata JSONL, và report JSON.


In [ ]:
RUN_FULL = False

if RUN_FULL:
    mapping_metadata, _ = load_category_mapping(MAPPING_PATH)
    if mapping_metadata.get("status") != "frozen":
        raise RuntimeError(
            "Mapping vẫn là draft. Hãy review mapping và đổi status='frozen' trước full run."
        )

    full_reports = {}
    for split in ("train", "valid", "test"):
        positive_path = CORE7_OUTPUT_DIR / f"category_clean_{split}.jsonl"
        metadata_path = CORE7_OUTPUT_DIR / f"core7_item_metadata_v1_{split}.jsonl"
        report_path = CORE7_OUTPUT_DIR / f"category_clean_{split}_report.json"

        report = prepare_clean_positive_split(
            split=split,
            output_path=positive_path,
            item_metadata_output_path=metadata_path,
            mapping_path=MAPPING_PATH,
            min_items=MIN_ITEMS,
            debug_limit=None,
        )
        with report_path.open("w", encoding="utf-8") as stream:
            json.dump(report, stream, ensure_ascii=False, indent=2)
            stream.write("\n")

        full_reports[split] = report

    print("FULL RUN COMPLETE")
    for split, report in full_reports.items():
        print(
            split,
            "| outfits:", report["outfits"]["outfits_kept"],
            "| metadata items:", report["item_metadata_validation"]["item_count"],
            "| metadata version:", report["item_metadata_version"],
        )
else:
    print("RUN_FULL=False — mới chỉ chạy debug, chưa tạo full Core-7 artifacts.")

## 11. Output contract của NB2

Sau full run, expected files:

```text
category_clean_train.jsonl
category_clean_valid.jsonl
category_clean_test.jsonl

core7_item_metadata_v1_train.jsonl
core7_item_metadata_v1_valid.jsonl
core7_item_metadata_v1_test.jsonl

category_clean_train_report.json
category_clean_valid_report.json
category_clean_test_report.json
```

Invariant quan trọng:

```text
set(all item_id trong category_clean_{split})
==
set(item_id trong core7_item_metadata_v1_{split})
```

Như vậy `coarse_category` được tạo ở category-drop stage không còn bị mất trước khi serialize.


## 12. Việc tiếp theo — trước negative sampler mới

1. kiểm tra/decode ảnh hoặc đối chiếu FashionCLIP embedding cache;
2. drop item thiếu ảnh/embedding và đếm lại outfit, giữ length >= 3;
3. filter Core-7 metadata theo **exact item IDs còn lại trong final clean positives**;
4. freeze final clean positives + matching item metadata theo version;
5. negative sampler mới đọc hai artifact này;
6. build candidate pools từ metadata (`master_category`, có thể thêm `coarse_category` cho experiment sau);
7. replacement phải cùng split, khác item, khác source kit và không có sẵn trong outfit;
8. tạo lại `swapped_item_index` trên final clean outfit;
9. merge positive + negative thành scorer dataset.

> Không dùng lại negative JSONL cũ sau category/image/embedding cleaning. Không dùng metadata artifact cũ nếu final clean step đã loại thêm item mà không filter metadata tương ứng.
